# real time face detecation happy ya sad

In [10]:
%pip install opencv-python==4.11.0.86

   ---------------------------------------- 0.0/39.5 MB ? eta -:--:--
   --- ------------------------------------ 3.7/39.5 MB 19.3 MB/s eta 0:00:02
   ------- -------------------------------- 7.3/39.5 MB 18.9 MB/s eta 0:00:02
   ----------- ---------------------------- 11.0/39.5 MB 18.2 MB/s eta 0:00:02
   -------------- ------------------------- 14.4/39.5 MB 17.8 MB/s eta 0:00:02
   ----------------- ---------------------- 17.6/39.5 MB 17.8 MB/s eta 0:00:02
   -------------------- ------------------- 19.9/39.5 MB 17.1 MB/s eta 0:00:02
   ---------------------- ----------------- 22.5/39.5 MB 15.7 MB/s eta 0:00:02
   -------------------------- ------------- 26.2/39.5 MB 16.0 MB/s eta 0:00:01
   ------------------------------ --------- 29.6/39.5 MB 15.9 MB/s eta 0:00:01
   -------------------------------- ------- 31.7/39.5 MB 15.4 MB/s eta 0:00:01
   ----------------------------------- ---- 34.6/39.5 MB 15.2 MB/s eta 0:00:01
   ---------------------------------------  38.8/39.5 MB 15.6 M

In [1]:
import cv2

print("Version:", cv2.__version__)
print("Location:", cv2.__file__)
print("CascadeClassifier:", hasattr(cv2, "CascadeClassifier"))

Version: 4.11.0
Location: c:\Users\Dell\AppData\Local\Programs\Python\Python314\Lib\site-packages\cv2\__init__.py
CascadeClassifier: True


In [2]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

print("Face Detector:", not face_cascade.empty())

Face Detector: True


In [3]:
from tensorflow.keras.models import load_model

model = load_model(
    r"E:\full stack data science\deep learning\CNN\Emotion Detection\emotion_model.keras"
)

print("Model loaded!")
print("Input:", model.input_shape)
print("Output:", model.output_shape)

Model loaded!
Input: (None, 200, 200, 3)
Output: (None, 1)


In [4]:

import cv2
import numpy as np
from tensorflow.keras.models import load_model

# 1. Load trained CNN model
model = load_model(
    r"E:\full stack data science\deep learning\CNN\Emotion Detection\emotion_model.keras"
)

print("CNN Model Loaded Successfully!")
print("Input Shape:", model.input_shape)
print("Output Shape:", model.output_shape)

# 2. Load face detection classifier
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

if face_cascade.empty():
    raise RuntimeError("Face detection model could not load!")

# 3. Open webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Webcam not opening!")

print("Camera Started! Press Q to quit.")

# 4. Real-time prediction
while True:

    ret, frame = cap.read()

    if not ret:
        print("Unable to read camera frame")
        break

    # Mirror camera
    frame = cv2.flip(frame, 1)

    # Convert to grayscale for face detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(50, 50)
    )

    # Process every detected face
    for (x, y, w, h) in faces:

        face = frame[y:y+h, x:x+w]

        # Preprocessing
        face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
        face = cv2.resize(face, (200, 200))
        face = face.astype("float32") / 255.0
        face = np.expand_dims(face, axis=0)

        # CNN Prediction
        prediction = model.predict(face, verbose=0)[0][0]

        # Classify emotion
        # Assumes class_indices: Happy=0, Sad=1
        if prediction >= 0.5:

            emotion = "Sad"
            confidence = prediction * 100
            color = (0, 0, 255)

        else:

            emotion = "Happy"
            confidence = (1 - prediction) * 100
            color = (0, 255, 0)

        # Draw face rectangle
        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            color,
            2
        )

        # Display emotion
        text = f"{emotion}: {confidence:.2f}%"

        cv2.putText(
            frame,
            text,
            (x, max(y-10, 25)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            color,
            2
        )

    # Show output
    cv2.imshow("CNN Real-Time Emotion Detection", frame)

    # Press Q to exit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# 5. Release resources
cap.release()
cv2.destroyAllWindows()

print("Emotion Detection Stopped!")


CNN Model Loaded Successfully!
Input Shape: (None, 200, 200, 3)
Output Shape: (None, 1)
Camera Started! Press Q to quit.
Emotion Detection Stopped!
